In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Paramètres couramment utilisés pour la transpilation

<details>
<summary><b>Versions des packages</b></summary>

Le code de cette page a été développé avec les dépendances suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Cette page décrit certains des paramètres les plus couramment utilisés pour la transpilation locale. Ces paramètres sont configurés via des arguments passés à [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) ou à [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<span id="approx-degree"></span>
## Degré d'approximation
Tu peux utiliser le degré d'approximation pour indiquer à quel point tu veux que le circuit résultant corresponde au circuit souhaité (en entrée). Il s'agit d'un flottant compris dans l'intervalle (0.0 - 1.0), où 0.0 correspond à une approximation maximale et 1.0 (valeur par défaut) correspond à aucune approximation. Des valeurs plus faibles sacrifient la précision de sortie au profit de la facilité d'exécution (c'est-à-dire moins de portes). La valeur par défaut est 1.0.

Lors de la synthèse unitaire à deux qubits (utilisée dans les étapes initiales de tous les niveaux et pour l'étape d'optimisation avec le niveau 3), cette valeur spécifie la fidélité cible de la décomposition en sortie. Autrement dit, quelle quantité d'erreur est introduite lorsqu'une représentation matricielle d'un circuit est convertie en portes discrètes. Si le degré d'approximation est faible (plus d'approximation), le circuit de sortie issu de la synthèse différera davantage de la matrice d'entrée, mais aura aussi probablement moins de portes (car toute opération arbitraire à deux qubits peut être décomposée parfaitement avec au plus trois portes CX) et sera plus facile à exécuter.

Lorsque le degré d'approximation est inférieur à 1.0, des circuits avec une ou deux portes CX peuvent être synthétisés, ce qui entraîne moins d'erreurs liées au matériel, mais davantage liées à l'approximation. Étant donné que CX est la porte la plus coûteuse en termes d'erreur, il peut être avantageux de réduire leur nombre au détriment de la fidélité lors de la synthèse (cette technique a été utilisée pour augmenter le volume quantique sur les appareils IBM&reg; : [Validating quantum computers using randomized model circuits](https://arxiv.org/abs/1811.12926)).

À titre d'exemple, nous générons un `UnitaryGate` aléatoire à deux qubits qui sera synthétisé lors de l'étape initiale. Définir `approximation_degree` à une valeur inférieure à 1.0 peut générer un circuit approximatif. Nous devons également spécifier les `basis_gates` pour indiquer à la méthode de synthèse quelles portes elle peut utiliser pour la synthèse approximative.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import random_unitary
from qiskit.transpiler import generate_preset_pass_manager

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qubits = QuantumRegister(2, name="q")
qc = QuantumCircuit(qubits)
qc.append(rand_U, qubits)
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    approximation_degree=0.85,
    basis_gates=["sx", "rz", "cx"],
)
approx_qc = pass_manager.run(qc)
print(approx_qc.count_ops()["cx"])

2


This yields an output of `2` because the approximation requires fewer CX gates.

<span id="seed"></span>
## Random number generator seed

Some parts of the transpiler are stochastic, so repeated transpilation runs may return different results. To obtain a reproducible result, you can set the seed for the pseudorandom number generator using the `seed_transpiler` argument. Repeated runs using the same seed will return the same results.

Example:

In [2]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, seed_transpiler=11, basis_gates=["sx", "rz", "cx"]
)
optimized_1 = pass_manager.run(qc)
optimized_1.draw("mpl")

<Image src="../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg" alt="Output of the previous code cell" />

Cela donne un résultat de `2` car l'approximation nécessite moins de portes CX.

<span id="seed"></span>
## Graine du générateur de nombres aléatoires
Certaines parties du Transpiler sont stochastiques, de sorte que des exécutions de transpilation répétées peuvent retourner des résultats différents. Pour obtenir un résultat reproductible, tu peux définir la graine du générateur de nombres pseudoaléatoires à l'aide de l'argument `seed_transpiler`. Des exécutions répétées avec la même graine retourneront les mêmes résultats.

Exemple :

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler import Layout

backend = FakeSherbrooke()

a, b = qubits
initial_layout = Layout({a: 5, b: 6})

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg)

<span id="init-layout"></span>
## Layout initial
Avant la transpilation, les qubits contenus dans ton circuit sont des qubits virtuels qui ne correspondent pas nécessairement à des qubits physiques du Backend cible. Tu peux spécifier la correspondance initiale entre qubits virtuels et qubits physiques à l'aide de l'argument `initial_layout`. Note que le layout final peut différer du layout initial, car le Transpiler peut permuter les qubits à l'aide de portes swap ou d'autres moyens.

Dans l'exemple ci-dessous, nous construisons un layout initial pour le Backend simulé [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) en créant un objet [`Layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Layout). Notre layout associe le premier qubit de notre circuit au qubit 5 de Sherbrooke, et le second qubit de notre circuit au qubit 6 de Sherbrooke. Note que les qubits physiques sont toujours représentés par des entiers.

In [4]:
initial_layout = [5, 6]

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg)

En plus de spécifier un objet Layout, tu peux également passer une liste d'entiers, où le $i$-ème élément de la liste contient le qubit physique sur lequel le $i$-ème qubit doit être mappé. Par exemple :

In [5]:
from qiskit.visualization import plot_error_map

plot_error_map(backend, figsize=(30, 24))

<Image src="../docs/images/guides/common-parameters/extracted-outputs/8df57c6a-1ff4-4d58-9b7e-4378452c3025-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg)

Tu peux utiliser la fonction [`plot_error_map`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.plot_error_map) pour générer un diagramme du graphe du dispositif avec les informations d'erreur et les qubits physiques étiquetés. Tu peux également consulter des diagrammes similaires sur la page [Compute resources](https://quantum.cloud.ibm.com/computers).